In [1]:
# =============================================================
# CELL 1 — CONFIG + INPUT AUDIT + LOAD          [FOLD-SAFE, v3]
#   Same pipeline both runs. Organizers change ONLY TEST_PATH.
# =============================================================
import glob, os, json, sys
import numpy as np, pandas as pd

TEST_PATH = "/kaggle/input/datasets/rafiurrahman01/bhd-all-inputs/rafiurrahman01/bengali-hallucination/test set.csv"          # <-- the only knob

def find(p):
    h = (glob.glob(f"/kaggle/input/**/{p}", recursive=True)
         + glob.glob(f"/kaggle/working/{p}"))
    return h[0] if h else None

REQUIRED = ["dataset samples.json", "bagdhara_pool.json", "tydiqa_bn_goldp.parquet",
            "bangla_mmlu_all.parquet", "qa_4000.csv", "qa_gt_1000.csv",
            "banglahallueval_qa_dataset.csv", "merged_pattern_dataset.csv",
            "passages.parquet", "bnwiki_e5_emb.npy"]
print("="*72); print("REQUIRED"); print("="*72)
missing = [p for p in REQUIRED if not find(p)]
for p in REQUIRED:
    h = find(p); print(f"  {'OK ' if h else 'XX '} {p:32s} {h or 'MISSING'}")
assert not missing, f"ATTACH FIRST: {missing}"

D14 = None
for d in sorted(glob.glob("/kaggle/input/**/qwen2.5-14b-instruct*/**/", recursive=True)):
    if "awq" in d.lower(): continue
    if glob.glob(os.path.join(d, "*.safetensors")): D14 = d.rstrip("/"); break
print(f"\n  {'OK ' if D14 else 'XX '} 14B fp16 judge : {D14}")
assert D14, "attach dipamc77/qwen2.5-14b-instruct (fp16, NOT awq)"

E5_DIR = None
for f in glob.glob("/kaggle/input/**/sentence_bert_config.json", recursive=True):
    E5_DIR = os.path.dirname(f); break
print(f"  {'OK ' if E5_DIR else 'XX '} E5 local dir   : {E5_DIR}")
assert E5_DIR, "attach the plain multilingual-e5-base dataset (internet is OFF)"
assert not any(k in E5_DIR.lower() for k in ("checkpoint","winner","hardneg","inbatch")), \
    f"E5 dir is a fine-tuned checkpoint, not the base model: {E5_DIR}"

tp = TEST_PATH or find("test set.csv") or find("test_set.csv")
assert tp, "no test file — set TEST_PATH"
print(f"\n  TEST FILE: {tp}")

def norm_df(df):
    df = df.copy()
    for c in ("context", "prompt_bn", "response_bn"):
        if c in df:
            df[c] = df[c].fillna("[NULL]" if c == "context" else "").astype(str)
    df["context"] = df["context"].str.strip().replace({"": "[NULL]", "nan": "[NULL]"})
    df["has_ctx"] = (df["context"] != "[NULL]").astype(int)
    return df

test = pd.read_csv(tp)
need = {"id", "context", "prompt_bn", "response_bn"}
assert need <= set(test.columns), f"missing cols: {need - set(test.columns)}"
assert test.id.is_unique, "duplicate ids in test file"
test = norm_df(test.sort_values("id").reset_index(drop=True))
sample = norm_df(pd.DataFrame(json.load(open(find("dataset samples.json"), encoding="utf-8"))))
y = sample["label"].values

print(f"  test  : {test.shape}  has_ctx={int(test.has_ctx.sum())} "
      f"no_ctx={int((1-test.has_ctx).sum())}  max_ctx={test.context.str.len().max()}")
print(f"  anchor: {sample.shape}  halluc_rate={(y==0).mean():.4f}")
print(f"  judge budget: {(len(test)+len(sample))*2.10/3600:.2f} h")

REF, PHASE1 = None, False
print("\n  MODE: FOLD-SAFE (identical pipeline both runs, pins off)")
print("\nCELL 1 PASSED.")

REQUIRED
  OK  dataset samples.json             /kaggle/input/datasets/rafiurrahman01/bhd-all-inputs/rafiurrahman01/bengali-hallucination/dataset samples.json
  OK  bagdhara_pool.json               /kaggle/input/datasets/rafiurrahman01/bhd-all-inputs/rafiurrahman01/bagdhara-pool-json/bagdhara_pool.json
  OK  tydiqa_bn_goldp.parquet          /kaggle/input/datasets/rafiurrahman01/bhd-all-inputs/rafiurrahman01/files2-zip/tydiqa_bn_goldp.parquet
  OK  bangla_mmlu_all.parquet          /kaggle/input/datasets/rafiurrahman01/bhd-all-inputs/rafiurrahman01/files2-zip/bangla_mmlu_all.parquet
  OK  qa_4000.csv                      /kaggle/input/datasets/rafiurrahman01/bhd-all-inputs/rafiurrahman01/banglahallueval-eb77-zip/Hallucination Generated Answers/qa_4000.csv
  OK  qa_gt_1000.csv                   /kaggle/input/datasets/rafiurrahman01/bhd-all-inputs/rafiurrahman01/banglahallueval-eb77-zip/QA/qa_gt_1000.csv
  OK  banglahallueval_qa_dataset.csv   /kaggle/input/datasets/rafiurrahman01/bhd-all-i

In [2]:
# =============================================================
# CELL 2 — NORMALIZERS + RULE LAYERS + GATES     [unchanged logic]
# =============================================================
import re, unicodedata
import numpy as np, pandas as pd
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BN2AR = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_ZW = ("\u200c", "\u200d", "\ufeff", "\u00a0")
def bn_norm(x):
    x = unicodedata.normalize("NFC", str(x))
    for z in _ZW: x = x.replace(z, "")
    x = x.translate(BN2AR)
    x = "".join(ch if (ch.isalnum() or ch.isspace()) else " " for ch in x)
    return re.sub(r"\s+", " ", x).strip().lower()
def keyify2(s):
    s = unicodedata.normalize("NFC", str(s)).translate(BN2AR)
    s = re.sub(r"[^\w\u0980-\u09FF]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()
def nfc(s):    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s))).strip()
def squash(s): return re.sub(r"[^\w\u0980-\u09FF]+", "", nfc(s))
def wtoks(s):  return [t for t in re.sub(r"[^\w\u0980-\u09FF]+", " ", nfc(s)).split() if t]
def toks(s):   return [t for t in keyify2(s).split() if t]
def sub_ok(a, b):
    a, b = bn_norm(a), bn_norm(b)
    return a == b if (len(a) < 3 or len(b) < 3) else (a in b or b in a)
def span_in(needle, hay):
    n, h = toks(needle), toks(hay)
    if not n or len(n) > len(h): return False
    return any(h[i:i+len(n)] == n for i in range(len(h)-len(n)+1))
def tokf1(a, b):
    A, B = set(toks(a)), set(toks(b))
    if not A or not B: return 0.0
    i = len(A & B)
    if not i: return 0.0
    p, r = i/len(B), i/len(A)
    return 2*p*r/(p+r)

_QUOTE = "'\u2018\u2019\"\u201c\u201d।,.()[]{}"
_U = {'শূন্য':0,'এক':1,'দুই':2,'দু':2,'তিন':3,'চার':4,'পাঁচ':5,'পাচ':5,'ছয়':6,'সাত':7,
 'আট':8,'নয়':9,'দশ':10,'এগারো':11,'এগার':11,'বারো':12,'বার':12,'তেরো':13,'তের':13,
 'চৌদ্দ':14,'পনেরো':15,'পনের':15,'ষোল':16,'ষোলো':16,'সতেরো':17,'সতের':17,
 'আঠারো':18,'আঠার':18,'উনিশ':19,'ঊনিশ':19,'বিশ':20,'একুশ':21,'বাইশ':22,'তেইশ':23,
 'চব্বিশ':24,'পঁচিশ':25,'ছাব্বিশ':26,'সাতাশ':27,'আটাশ':28,'ঊনত্রিশ':29,'ত্রিশ':30,
 'একত্রিশ':31,'বত্রিশ':32,'তেত্রিশ':33,'চৌত্রিশ':34,'পঁয়ত্রিশ':35,'ছত্রিশ':36,
 'চল্লিশ':40,'পঞ্চাশ':50,'ষাট':60,'সত্তর':70,'আশি':80,'নব্বই':90,
 'অষ্টাশি':88,'একশত':100,'একশ':100,'শতাধিক':100}
_S = {'শত':100,'শো':100,'শ':100,'হাজার':1000,'হাযার':1000,'লক্ষ':10**5,'লাখ':10**5,
      'মিলিয়ন':10**6,'নিযুত':10**6,'কোটি':10**7,'বিলিয়ন':10**9}
_ORD   = re.compile(r'(?<=[০-৯0-9])\s*(রা|লা|শে|ই|ঠা|য়ে|তম)\b')
_GEN   = re.compile(r'(ের|র|ে|য়|কে|তে)$')
_SPLIT = re.compile(r'\d[-/:.]\d')
ACRO  = {'বিএনপি':'বাংলাদেশ জাতীয়তাবাদী দল','আওয়ামীলীগ':'বাংলাদেশ আওয়ামী লীগ',
         'ঢাবি':'ঢাকা বিশ্ববিদ্যালয়','বুয়েট':'বাংলাদেশ প্রকৌশল বিশ্ববিদ্যালয়'}
MONTH = {'জানুয়ারী':'জানুয়ারি','ফেব্রুয়ারী':'ফেব্রুয়ারি','অগস্ট':'আগস্ট','আগষ্ট':'আগস্ট'}
def _clean(t): return t.strip(_QUOTE)
def _tokval(t):
    t = _clean(t)
    if t in _U: return ('u', float(_U[t]))
    if t in _S: return ('s', float(_S[t]))
    if re.fullmatch(r'\d+(?:\.\d+)?', t): return ('u', float(t))
    for suf in ('শত','শো','শ'):
        if t.endswith(suf) and t[:-len(suf)] in _U:
            return ('us', float(_U[t[:-len(suf)]]) * 100.0)
    return None
def _chunk(vals):
    tot = cur = 0.0
    for k, v in vals:
        if k in ('u','us'): cur += v
        else:
            if v <= 100: cur = (cur or 1.0) * v
            else: tot += (cur or 1.0) * v; cur = 0.0
    return tot + cur
def numvals(s):
    s = unicodedata.normalize("NFC", str(s)).translate(BN2AR); s = _ORD.sub('', s)
    out, chunk = set(), []
    for raw in s.split():
        if _SPLIT.search(raw):
            if chunk: out.add(_chunk(chunk)); chunk = []
            for d in re.findall(r'\d+(?:\.\d+)?', raw): out.add(float(d))
            continue
        tv = _tokval(raw)
        if tv: chunk.append(tv)
        elif _clean(raw) in ('ও','এবং','আর') and chunk: continue
        else:
            if chunk: out.add(_chunk(chunk)); chunk = []
            for d in re.findall(r'\d+(?:\.\d+)?', raw): out.add(float(d))
    if chunk: out.add(_chunk(chunk))
    return {round(v, 6) for v in out if v}
def deep_norm(x):
    x = unicodedata.normalize("NFC", str(x))
    for z in _ZW: x = x.replace(z, "")
    x = x.translate(BN2AR); x = _ORD.sub('', x)
    for k, v in MONTH.items(): x = x.replace(k, v)
    tk = [t for t in re.split(r'[^\w\u0980-\u09FF]+', x) if t]
    tk = [ACRO.get(t, t) for t in tk]
    tk = [_GEN.sub('', t) if len(t) > 4 else t for t in tk]
    return " ".join(tk).lower().strip()
def sub_ok2(a, b):
    na, nb = numvals(a), numvals(b)
    if na and nb:
        if na & nb: return True
        if not (na <= nb or nb <= na): return False
    A, B = deep_norm(a), deep_norm(b)
    if len(A) < 3 or len(B) < 3: return A == B
    return A in B or B in A

CASES = [("৩","তিন",True), ("আঠারশ' বত্রিশ সালে","1832",True),
         ("২রা নভেম্বর, ১৮৮৬","২ নভেম্বর ১৮৮৬",True),
         ("২ জানুয়ারি ১৯৭১","২ জানুয়ারী, ১৯৭১",True),
         ("রংপুর জিলা স্কুল","রংপুর জিলা স্কুলের",True),
         ("বিএনপি","বাংলাদেশ জাতীয়তাবাদী দলের",True),
         ("প্রায় ১ কোটি","প্রায় ১০ মিলিয়ন",True),
         ("দুই লক্ষ তেরো হাজার দুই শত","213200",True),
         ("১০১টি গ্রাম","১১০",False), ("৭৭৪","৭৭৬",False),
         ("ষোল ফুট","১০ ফুট",False), ("২০০৫","১৯৪৮",False),
         ("২০১৯-২০","2039",False)]
bad = sum(sub_ok2(a,b) != w for a,b,w in CASES)
print(f"sub_ok2 unit tests: {len(CASES)-bad}/{len(CASES)} pass")
assert not bad, "sub_ok2 SUITE FAILED"

ty     = pd.read_parquet(find("tydiqa_bn_goldp.parquet"))
mmlu   = pd.read_parquet(find("bangla_mmlu_all.parquet"))
idioms = json.load(open(find("bagdhara_pool.json"), encoding="utf-8"))
qa4    = pd.read_csv(find("qa_4000.csv"))
qagt   = pd.read_csv(find("qa_gt_1000.csv"))
qads   = pd.read_csv(find("banglahallueval_qa_dataset.csv"))

ty["gold"] = ty["answers"].map(lambda a: str(a["text"][0]).strip() if len(a["text"]) else "")
ty = ty[ty.gold.str.len() > 0]
TY_Q = {}
for _, r in ty.iterrows(): TY_Q.setdefault(bn_norm(r["question"]), r["gold"])
LET = {"A":0,"B":1,"C":2,"D":3,"E":4}
def _mg(row):
    ch = list(row["choices"]); i = LET.get(str(row["answer"]).strip().upper())
    return ch[i] if i is not None and i < len(ch) else None
mmlu["gold"] = mmlu.apply(_mg, axis=1)
MMLU_Q = {}
for _, r in mmlu.iterrows():
    if r["gold"] is not None:
        MMLU_Q.setdefault(bn_norm(r["question"]), (r["gold"], list(r["choices"])))
IDIOM = {bn_norm(k): v for k, v in idioms.items()}
_iv = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4))
print(f"TY_Q={len(TY_Q)}  MMLU_Q={len(MMLU_Q)}  IDIOM={len(IDIOM)}")

def quoted_hw(p):
    m = re.search(r"[\"\u201c'\u2018]([^\"\u201d'\u2019]+)[\"\u201d'\u2019]", str(p))
    if m: return m.group(1).strip()
    m = re.search(r"([\u0980-\u09FF][\u0980-\u09FF\s\-]{1,25}?)'\s*(শব্দ|বাগধারা|প্রবাদ)", str(p))
    return m.group(1).strip() if m else None
_D  = r"[০-৯0-9]"
_MO = (r"(?:জানুয়ারি|জানুয়ারী|ফেব্রুয়ারি|ফেব্রুয়ারী|মার্চ|এপ্রিল|মে|জুন|জুলাই|"
       r"আগস্ট|অগস্ট|সেপ্টেম্বর|অক্টোবর|নভেম্বর|ডিসেম্বর)")
_DT = (rf"(?:{_D}{{1,2}}(?:লা|রা|শে|ই|ঠা)?\s*{_MO}[,\s]+{_D}{{4}}"
       rf"|{_MO}\s*{_D}{{1,2}}[,\s]+{_D}{{4}}|{_D}{{4}})")
_BP = [rf"জন্ম[\s:ঃ]*[\(]?\s*({_DT})", rf"({_D}{{4}})\s*সালের[^।]{{0,50}}?জন্মগ্রহণ",
       rf"^[^(]{{1,40}}\(\s*({_DT})\s*[-–—]"]
def extract_birth(c):
    c = unicodedata.normalize("NFC", str(c)).replace("\u200c","").replace("\u200d","")
    for p in _BP:
        m = re.search(p, c)
        if m: return m.group(1)
    return None
def is_when_birth(q):
    q = str(q)
    if "কোথায়" in q or re.search(r"\bকে\b", q): return False
    return bool(re.search(r"(কবে|কত সালে|কোন সালে|কোন তারিখে|কত তারিখে|কত খ্রিস্টাব্দে)", q)
                and "জন্ম" in q)
def subject_in_ctx(q, ctx):
    stem = re.split(r"(কত সালে|কবে|কোন সালে|কোন তারিখে|কত তারিখে)", str(q))[0].strip()
    t = [x for x in stem.split()[:2] if len(x) > 2]
    return (not t) or any(x in str(ctx) for x in t)
def _idiom_pred(resp, meanings, thr=0.35):
    tx = [bn_norm(resp)] + [bn_norm(x) for x in meanings]
    try:
        X = _iv.fit(tx).transform(tx)
        return 1 if cosine_similarity(X[0], X[1:]).max() >= thr else 0
    except Exception:
        return None

WEAK   = {"ctx_substr", "mmlu_nomatch"}
WEAK_A = WEAK | {"birth", "idiom", "ty_noctx"}

def label_row3(q, resp, ctx, has_ctx):
    qn = bn_norm(q)
    if has_ctx:
        if is_when_birth(q) and subject_in_ctx(q, ctx):
            bd = extract_birth(ctx)
            if bd is not None: return (1 if sub_ok2(resp, bd) else 0, "birth")
        if qn in TY_Q: return (1 if sub_ok2(resp, TY_Q[qn]) else 0, "ty_gold")
        if qn in MMLU_Q:
            g, ch = MMLU_Q[qn]
            if sub_ok(resp, g): return (1, "mmlu_gold")
            for c in ch:
                if c != g and sub_ok(resp, c): return (0, "mmlu_distractor")
            return (0, "mmlu_nomatch")
        if re.search(r"ভাবার্থ|বাগধারা|প্রবাদ", str(q)) and "শাব্দিক" not in str(q):
            hw = quoted_hw(q)
            if hw and bn_norm(hw) in IDIOM:
                p = _idiom_pred(resp, IDIOM[bn_norm(hw)])
                if p is not None: return (p, "idiom")
        if sub_ok(resp, ctx): return (1, "ctx_substr")
        return (None, None)
    if qn in MMLU_Q:
        g, ch = MMLU_Q[qn]
        if sub_ok(resp, g): return (1, "mmlu_gold")
        for c in ch:
            if c != g and sub_ok(resp, c): return (0, "mmlu_distractor")
        return (0, "mmlu_nomatch")
    if qn in TY_Q: return (1 if sub_ok2(resp, TY_Q[qn]) else 0, "ty_noctx")
    if re.search(r"ভাবার্থ|বাগধারা|প্রবাদ", str(q)) and "শাব্দিক" not in str(q):
        hw = quoted_hw(q)
        if hw and bn_norm(hw) in IDIOM:
            p = _idiom_pred(resp, IDIOM[bn_norm(hw)])
            if p is not None: return (p, "idiom")
    return (None, None)

def qn2(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", s)
    s = re.sub(r"[\u0964।,.!?;:\"'()\[\]{}*_`-]", "", s)
    return re.sub(r"\s+", " ", s).strip().lower()
gold, hallu, model_a = defaultdict(set), defaultdict(set), defaultdict(list)
for _, r in qa4.iterrows():
    k = qn2(r["question"]); gold[k].add(qn2(r["right_answer"])); hallu[k].add(qn2(r["hallucinated_answer"]))
for _, r in qagt.iterrows(): gold[qn2(r["question"])].add(qn2(r["correct_answer"]))
for _, r in qads.iterrows():
    k = qn2(r["question"]); gold[k].add(qn2(r["correct_answer"]))
    for mm in ("deepseek","gemma","qwen"):
        a, s = r.get(f"{mm}_answer"), r.get(f"{mm}_score")
        if pd.notna(a) and pd.notna(s): model_a[k].append((qn2(a), int(s)))
gold = {k: {x for x in v if x} for k, v in gold.items()}
amb  = {k for k, v in gold.items() if len(v) > 1}
pois = {k for k in gold if gold[k] & hallu.get(k, set())}
print(f"gold key: {len(gold)} q | ambiguous {len(amb)} | poisoned {len(pois)}")
def goldkey(q, resp):
    k = qn2(q)
    if k not in gold or k in pois: return None, None
    r = qn2(resp)
    if r in hallu.get(k, set()):   return 0, "R2_exact_hallu"
    if r in gold[k]:               return 1, "R1_exact_gold"
    for a, s in model_a.get(k, []):
        if a and a == r and s == 1: return 1, "R5_model_correct"
    if k in amb:                   return None, None
    g = next(iter(gold[k]))
    if span_in(g, r) or span_in(r, g): return 1, "R3_span"
    return None, None

XREF = re.compile(r"সমার্থক\s*বাগধারা.*$")
def clean_meanings(vals):
    out = []
    for v in ([vals] if isinstance(vals, str) else list(vals)):
        v = XREF.sub("", str(v))
        for part in re.split(r"[;।]", v):
            part = keyify2(part)
            if len(part) >= 2: out.append(part)
    return out
POOL = {}
for k, v in idioms.items():
    ms = clean_meanings(v)
    if not ms: continue
    for variant in re.split(r"/", str(k)):
        vk = keyify2(variant)
        if vk: POOL.setdefault(vk, set()).update(ms)
print(f"POOL: {len(idioms)} raw -> {len(POOL)} variants")
def idiom_key(p):
    m = re.search(r"[\"\u201c\u201d'\u2018\u2019](.*?)[\"\u201c\u201d'\u2018\u2019]", str(p))
    return keyify2(m.group(1)) if m else None
def ptype(p):
    p = str(p)
    if "শাব্দিক অর্থ" in p: return "lit"
    if "ভাবার্থ"      in p: return "fig"
    return "ctx"
HI, LO = 0.70, 0.00
def idiom_fig(q, resp):
    if ptype(q) != "fig": return None, None
    k = idiom_key(q)
    if not k or k not in POOL: return None, None
    ms = POOL[k]
    f1_ = max(tokf1(resp, x) for x in ms)
    con = any(span_in(x, resp) or span_in(resp, x) for x in ms)
    if con or f1_ >= HI: return 1, "idiom_fig"
    if f1_ <= LO:        return 0, "idiom_fig"
    return None, None

NEG   = {"না", "নাই", "নয়", "নেই"}
AVYAY = {"অনুযায়ী","ব্যাপিয়া","ধরিয়া","পর্যন্ত","যথা","প্রতি","অনু","উপ","সমীপে","অভাবে"}
BAHU  = {"যার", "যাহার", "যাদের", "যাঁর"}
SIM   = ("ন্যায়", "মত", "সদৃশ")
def samas_type(bv):
    t = wtoks(bv); s = nfc(bv)
    if not t: return None
    if "পরস্পর" in t:                          return "ব্যতিহার বহুব্রীহি"
    if any(w in BAHU for w in t):
        return "নঞ বহুব্রীহি" if any(w in NEG for w in t) else "বহুব্রীহি"
    if "সমাহার" in t:                          return "দ্বিগু"
    if {"ও", "এবং", "আর"} & set(t):
        if any(w.endswith("ে") for w in t):    return None
        return "দ্বন্দ্ব"
    if re.search(r"-ই", s) or "রূপ" in t:      return "রূপক কর্মধারয়"
    for k in SIM:
        if k in t:
            return "উপমিত কর্মধারয়" if t.index(k) == len(t) - 1 else "উপমান কর্মধারয়"
    if {"চিহ্নিত", "রাখা", "ভরা"} & set(t):    return "মধ্যপদলোপী কর্মধারয়"
    if "যে" in t:                              return "কর্মধারয়"
    if t[0] in NEG and len(t) == 2:            return "নঞ তৎপুরুষ"
    if "পর্যন্ত" in t or t[-1] in AVYAY or t[0] in AVYAY: return "অব্যয়ীভাব"
    if {"দ্বারা", "দিয়ে", "দিয়া", "কর্তৃক"} & set(t):    return "তৃতীয়া তৎপুরুষ"
    if {"জন্য", "নিমিত্ত"} & set(t):           return "চতুর্থী তৎপুরুষ"
    if {"হইতে", "হতে", "থেকে", "চেয়ে"} & set(t):        return "পঞ্চমী তৎপুরুষ"
    if "কে" in t:                              return "দ্বিতীয়া তৎপুরুষ"
    if re.search(r"(ের|র)\s", s + " "):        return "ষষ্ঠী তৎপুরুষ"
    if re.search(r"(ে|য়|তে)\s", s + " "):     return "সপ্তমী তৎপুরুষ"
    return None
def same_samas(pred, resp):
    p, r = wtoks(pred), wtoks(resp)
    if not p or not r: return None
    if p[-1] != r[-1]: return False
    if len(r) == 1:    return True
    return " ".join(p) == " ".join(r)
SAM_Q, SAN_Q = "ব্যাসবাক্য অনুযায়ী", "শব্দ দুটির সন্ধিতে"
BV_RE   = re.compile(r"ব্যাসবাক্য হলো[:ঃ]?\s*(.*?)\s*[।.]?\s*$")
PAIR_RE = re.compile(r"^(.*?)'?\s*ও\s*'(.*?)'\s*শব্দ")
def grammar(ctx, q, resp):
    c, p, r = nfc(ctx), nfc(q), nfc(resp)
    if SAM_Q in p:
        m = BV_RE.search(c)
        if not m: return None, None
        pred = samas_type(m.group(1))
        if pred is None: return None, None
        v = same_samas(pred, r)
        return (None, None) if v is None else (int(v), "samas")
    if SAN_Q in p:
        m = PAIR_RE.match(p)
        if not m: return None, None
        naive     = squash(m.group(1)) + squash(m.group(2))
        is_naive  = squash(r) == naive
        unchanged = "অপরিবর্তিত" in c
        return (int(is_naive) if unchanged else int(not is_naive)), "sandhi"
    return None, None

def stack3(ctx, q, resp, has_ctx, weak):
    v, t = grammar(ctx, q, resp)
    if v is not None: return v, t
    v, t = idiom_fig(q, resp)
    if v is not None: return v, t
    v, t = goldkey(q, resp)
    if v is not None: return v, t
    v, t = label_row3(q, resp, ctx, has_ctx)
    if v is not None and t not in weak: return v, t
    return None, None

# ---------------- GATES (deterministic: same 299 anchor both runs) ----------------
from sklearn.metrics import f1_score
print("\n" + "="*72); print("GATES"); print("="*72)
o = [label_row3(r.prompt_bn, r.response_bn, r.context, r.has_ctx) for r in sample.itertuples()]
l1 = pd.Series([x[0] for x in o]); r1 = pd.Series([x[1] for x in o])
sm = l1.notna() & ~r1.isin(WEAK)
acc1 = (l1[sm].values == y[sm.values]).mean()
print(f"  GATE1 rules : strong {int(sm.sum())}/{len(sample)}  acc={acc1:.4f}  (need >=0.93)")
assert acc1 >= 0.93, "GATE1 FAILED"
o = [goldkey(r.prompt_bn, r.response_bn) for r in sample.itertuples()]
l2 = pd.Series([x[0] for x in o]); c2 = l2.notna()
acc2 = (l2[c2].values == y[c2.values]).mean() if c2.sum() else 1.0
print(f"  GATE2 goldkey: fires {int(c2.sum())}  acc={acc2:.4f}  (need 1.0000)")
assert acc2 == 1.0, "GATE2 FAILED"
o = [idiom_fig(r.prompt_bn, r.response_bn) for r in sample.itertuples()]
l3 = pd.Series([x[0] for x in o]); c3 = l3.notna()
acc3 = (l3[c3].values == y[c3.values]).mean() if c3.sum() else float("nan")
print(f"  GATE3 idiom : fires {int(c3.sum())}  acc={acc3:.4f}  (need 1.0000)")
assert c3.sum() >= 8 and acc3 == 1.0, "GATE3 FAILED"

ot = [stack3(r.context, r.prompt_bn, r.response_bn, r.has_ctx, WEAK_A) for r in test.itertuples()]
RULE_T = np.array([x[0] if x[0] is not None else -1 for x in ot])
SRC_T  = np.array([x[1] or "judge" for x in ot])
oa = [stack3(r.context, r.prompt_bn, r.response_bn, r.has_ctx, WEAK_A) for r in sample.itertuples()]
RULE_A = np.array([x[0] if x[0] is not None else -1 for x in oa])
SRC_A  = np.array([x[1] or "judge" for x in oa])
print(f"\n  TEST  : rules {int((RULE_T>=0).sum())}/{len(test)}, judge {int((RULE_T<0).sum())}")
print(pd.Series(SRC_T[RULE_T>=0]).value_counts().to_string())
print(f"\n  ANCHOR: rules {int((RULE_A>=0).sum())}, rule acc="
      f"{(RULE_A[RULE_A>=0]==y[RULE_A>=0]).mean():.4f}")
if PHASE1:
    assert int((RULE_T>=0).sum()) == 1532, "Phase-1 rule coverage drifted"
print("\nCELL 2 PASSED.")

sub_ok2 unit tests: 13/13 pass
TY_Q=2469  MMLU_Q=82748  IDIOM=8519
gold key: 2478 q | ambiguous 15 | poisoned 0
POOL: 8791 raw -> 9983 variants

GATES
  GATE1 rules : strong 178/299  acc=0.9775  (need >=0.93)
  GATE2 goldkey: fires 101  acc=1.0000  (need 1.0000)
  GATE3 idiom : fires 10  acc=1.0000  (need 1.0000)

  TEST  : rules 1532/2516, judge 984
R1_exact_gold       553
R2_exact_hallu      351
mmlu_distractor     225
mmlu_gold           208
idiom_fig            68
samas                48
sandhi               42
ty_gold              35
R5_model_correct      1
R3_span               1

  ANCHOR: rules 178, rule acc=0.9888

CELL 2 PASSED.


In [3]:
# =============================================================
# CELL 3 — 14B MARGIN JUDGE (live, checkpointed)   ~90 min / 2516 rows
# =============================================================
import gc, time, os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

gc.collect(); torch.cuda.empty_cache()
L, SPILL = 48, 6
def make_map(spill):
    keep = L - spill; half = keep // 2
    dm = {"model.embed_tokens": 0, "model.rotary_emb": 0}
    for i in range(L):
        dm[f"model.layers.{i}"] = "cpu" if i >= keep else (0 if i < half else 1)
    dm["model.norm"] = 1; dm["lm_head"] = 1
    return dm
TOKZ = AutoTokenizer.from_pretrained(D14)
if TOKZ.pad_token is None: TOKZ.pad_token = TOKZ.eos_token
ID_H = TOKZ.encode("0", add_special_tokens=False)[0]
ID_F = TOKZ.encode("1", add_special_tokens=False)[0]
JUDGE = AutoModelForCausalLM.from_pretrained(
    D14, device_map=make_map(SPILL), dtype=torch.float16,
    offload_buffers=True, offload_folder="/kaggle/temp/off").eval()
print(f"JUDGE loaded | ids 0->{ID_H} 1->{ID_F}")

CTX_CAP, MAXLEN, THR = 1200, 2048, 0.340
SYS_A = ("তুমি একজন কঠোর বাংলা তথ্য-যাচাইকারী। প্রসঙ্গ থাকলে শুধু প্রসঙ্গের ভিত্তিতে, "
         "না থাকলে সাধারণ জ্ঞানে বিচার করো। উত্তরটি সঠিক ও সমর্থিত হলে লেখো 1, "
         "ভুল বা অসমর্থিত হলে লেখো 0। শুধু একটি অঙ্ক লিখবে, আর কিছু নয়।")
def judge_margin(ctx, q, a):
    for cap in (CTX_CAP, CTX_CAP // 3, 200):
        try:
            c = ctx if ctx != "[NULL]" else "(নেই)"
            user = f"প্রসঙ্গ: {c[:cap]}\nপ্রশ্ন: {q}\nউত্তর: {a}\nরায়:"
            txt = TOKZ.apply_chat_template(
                [{"role": "system", "content": SYS_A}, {"role": "user", "content": user}],
                add_generation_prompt=True, tokenize=False)
            enc = TOKZ(txt, return_tensors="pt", truncation=True, max_length=MAXLEN).to(0)
            with torch.no_grad():
                lg = JUDGE(**enc).logits[0, -1, :].float()
            return lg[ID_H].item() - lg[ID_F].item(), 1
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
    return 0.0, 0

def run_margins(df, tag):
    ck = f"/kaggle/working/margins_{tag}.npy"; ok = f"/kaggle/working/ok_{tag}.npy"
    if os.path.exists(ck):
        m, o = np.load(ck), np.load(ok); start = int((o >= 0).sum())
        if start >= len(df): print(f"{tag}: cached"); return m
    else:
        m, o = np.zeros(len(df)), np.full(len(df), -1); start = 0
    t0 = time.time()
    for i in range(start, len(df)):
        r = df.iloc[i]
        m[i], o[i] = judge_margin(r.context, r.prompt_bn, r.response_bn)
        if (i+1) % 200 == 0:
            torch.cuda.empty_cache(); np.save(ck, m); np.save(ok, o)
            el = time.time()-t0; rt = el/(i+1-start)
            print(f"  {tag} {i+1}/{len(df)}  {el/60:.1f}m  {rt:.2f}s/row  "
                  f"eta {(len(df)-i-1)*rt/60:.0f}m")
    np.save(ck, m); np.save(ok, o)
    print(f"{tag}: done in {(time.time()-t0)/60:.1f} min")
    return m

mgA = run_margins(sample, "anchor")
lr  = LogisticRegression(max_iter=1000).fit(mgA.reshape(-1,1), (y==0).astype(int))
COEF, INTC = float(lr.coef_[0][0]), float(lr.intercept_[0])
pa  = 1/(1+np.exp(-(COEF*mgA + INTC)))
ja  = (pa < THR).astype(int)
print(f"\nPlatt: coef={COEF:.6f} intc={INTC:.6f}")
print(f"anchor judge F1_0={f1_score(y, ja, pos_label=0):.4f} "
      f"AUC={roc_auc_score((y==0).astype(int), pa):.4f}   [Phase-1 reference: 0.7619 / 0.8407]")

tm  = run_margins(test, "test")
pt  = 1/(1+np.exp(-(COEF*tm + INTC)))
jt  = (pt < THR).astype(int)
print(f"judge-alone on test: zeros={int((jt==0).sum())}  [Phase-1 reference: 1509 on 2516 rows]")
print("CELL 3 DONE.")

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


JUDGE loaded | ids 0->15 1->16
  anchor 200/299  6.7m  2.00s/row  eta 3m
anchor: done in 10.4 min

Platt: coef=0.083657 intc=-0.407948
anchor judge F1_0=0.7619 AUC=0.8407   [Phase-1 reference: 0.7619 / 0.8407]
  test 200/2516  4.9m  1.46s/row  eta 56m
  test 400/2516  10.1m  1.51s/row  eta 53m
  test 600/2516  17.1m  1.71s/row  eta 55m
  test 800/2516  24.9m  1.87s/row  eta 53m
  test 1000/2516  32.5m  1.95s/row  eta 49m
  test 1200/2516  39.9m  1.99s/row  eta 44m
  test 1400/2516  47.3m  2.03s/row  eta 38m
  test 1600/2516  55.2m  2.07s/row  eta 32m
  test 1800/2516  63.0m  2.10s/row  eta 25m
  test 2000/2516  71.0m  2.13s/row  eta 18m
  test 2200/2516  77.8m  2.12s/row  eta 11m
  test 2400/2516  83.8m  2.09s/row  eta 4m
test: done in 88.3 min
judge-alone on test: zeros=1509  [Phase-1 reference: 1509 on 2516 rows]
CELL 3 DONE.


In [4]:
# =============================================================
# CELL 4 — MATH LAYERS: v1 + v2 + weekday + repair_general
# =============================================================
import re, unicodedata, numpy as np
from math import comb, gcd, sqrt, tan, pi
import sympy as sp

def _nf(s):
    s = unicodedata.normalize("NFC", str(s)).translate(BN2AR)
    s = s.replace("²","^2").replace("³","^3").replace("–","-").replace("—","-")
    s = re.sub(r"x\s*\^?\s*2", "x^2", s)
    s = re.sub(r"x\s*\^?\s*3", "x^3", s)
    return re.sub(r"\s+", " ", s)
def mnums(s):
    s = str(s).translate(BN2AR).replace(",", "")
    return [float(x) for x in re.findall(r"\d+(?:\.\d+)?", s)]
def _close(a,b): return abs(a-b) <= 1e-6*max(1,abs(b))
def _lcm(a,b): return a*b//gcd(int(a),int(b))

# ---- v1 (25 templates) ----
def _solve1(Q):
    Q=str(Q); n=mnums(Q)
    if "যথাক্রমে" in Q and "একা একা শেষ" in Q and len(n)>=3: a,b,c=n[:3]; return 1/(1/a+1/b+1/c)
    if "তিন ব্যবসায়িক অংশীদারের" in Q and "দ্বিতীয়" in Q and len(n)>=4: T,a,b,c=n[:4]; return T*b/(a+b+c)
    if "চিনি ও পানির অনুপাত" in Q and "পানি কত" in Q and len(n)>=3: s,w,T=n[:3]; return T*w/(s+w)
    if "একে অপরের দিকে" in Q and len(n)>=3: D,v1,v2=n[:3]; return D/(v1+v2)
    if "একই দিকে" in Q and "দূরত্ব" in Q and len(n)>=3: v1,v2,t=n[:3]; return abs(v1-v2)*t
    if "রহিম একা" in Q and "করিম একা" in Q and len(n)>=2: a,b=n[:2]; return 1/(1/a+1/b)
    if "একাই" in Q and "সমাপ্ত" in Q and len(n)>=2: a,b=n[:2]; return 1/(1/a+1/b)
    if "বেড়ে যায়" in Q and "কমে যায়" in Q and len(n)>=3: p,q,P=n[:3]; return P*(1+p/100)*(1-q/100)
    if "বৃদ্ধি করা হয়" in Q and "ছাড়" in Q and len(n)>=3: P,p,q=n[:3]; return P*(1+p/100)*(1-q/100)
    if "সরল সুদের হারে" in Q and "মোট সুদ" in Q and len(n)>=3: P,r,t=n[:3]; return P*r*t/100
    if "সরল সুদে ঋণ" in Q and "সুদ পাবেন" in Q and len(n)>=3: P,t,r=n[:3]; return P*r*t/100
    if ("সংকেত বাতি" in Q or "বাস স্টপেজ" in Q) and len(n)>=3: a,b,c=map(int,n[:3]); return _lcm(_lcm(a,b),c)
    if "গড়মান" in Q and "নতুন রাশি" in Q and len(n)>=3: N,m,k=n[:3]; return k*(N+1)-m*N
    if "গড় নম্বর" in Q and "নতুন একজন" in Q and len(n)>=3: N,m,k=n[:3]; return k*(N+1)-m*N
    if "মা ও মেয়ের বয়সের অনুপাত" in Q and "মেয়ের বয়স" in Q and len(n)>=3: a,b,S=n[:3]; return S*b/(a+b)
    if "ভাইয়ের বয়সের অনুপাত" in Q and "ছোট ভাইয়ের" in Q and len(n)>=3: a,b,S=n[:3]; return S*min(a,b)/(a+b)
    if "গরু ও ছাগলের সংখ্যার অনুপাত" in Q and "ছাগলের সংখ্যা" in Q and len(n)>=3: a,b,T=n[:3]; return T*b/(a+b)
    if "রুই ও কাতলা" in Q and "কাতলা মাছের সংখ্যা কত" in Q and len(n)>=3: a,b,T=n[:3]; return T*b/(a+b)
    if "ক্রয়মূল্য" in Q and "লাভে বিক্রি" in Q and len(n)>=2: C,p=n[:2]; return C*(1+p/100)
    if "ক্রয়মূল্য" in Q and "ক্ষতিতে বিক্রি" in Q and len(n)>=2: C,p=n[:2]; return C*(1-p/100)
    if re.search(r"পণ্য .* টাকায় কেনা",Q) and "লাভে বিক্রয়" in Q and len(n)>=2: C,p=n[:2]; return C*(1+p/100)
    if re.search(r"পণ্য .* টাকায় কেনা",Q) and "ক্ষতিতে বিক্র" in Q and len(n)>=2: C,p=n[:2]; return C*(1-p/100)
    if ("প্যানেল গঠন" in Q or "উপকমিটি গঠন" in Q) and len(n)>=2: N,r=map(int,n[:2]); return comb(N,r)
    return None
def solve_math(q,resp):
    try: v=_solve1(q)
    except Exception: return None
    if v is None: return None
    rn=mnums(resp)
    if not rn: return None
    return 1 if any(_close(x,v) for x in rn) else 0

# ---- v2 (sympy templates) ----
def _fr(s):
    s=_nf(s); out=set()
    for m in re.finditer(r"(\d+(?:\.\d+)?)?\s*√\s*(\d+(?:\.\d+)?)", s):
        c=float(m.group(1)) if m.group(1) else 1.0; out.add(c*sqrt(float(m.group(2))))
    for m in re.finditer(r"(?<![\d.])(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)(?![\d.])", s):
        out.add(float(m.group(1))/float(m.group(2)))
    for m in re.finditer(r"(?<![\d./√])(\d+(?:\.\d+)?)(?![\d./])", s): out.add(float(m.group(1)))
    return out
def _ok(v,resp,tol=1e-4): return any(abs(x-v)<=tol*max(1,abs(v)) for x in _fr(resp))
def _quad(qs):
    m=re.search(r"(-?\d*)\s*x\^2\s*([+-])\s*(\d+)\s*x\s*([+-])\s*(\d+)", qs)
    if not m: return None
    a=int(m.group(1)) if m.group(1) not in("","-") else(-1 if m.group(1)=="-" else 1)
    b=int(m.group(3))*(1 if m.group(2)=="+" else -1); c=int(m.group(5))*(1 if m.group(4)=="+" else -1)
    return a,b,c
def T_quad_ineq(q,r):
    qs=_nf(q)
    if "সমাধান" not in qs or "x^2" not in qs: return None
    m=re.search(r"([<>]=?|≤|≥)\s*0",qs); co=_quad(qs)
    if not(m and co): return None
    a,b,c=co; x=sp.symbols("x",real=True); op=m.group(1)
    rel={"<":sp.Lt,"<=":sp.Le,"≤":sp.Le,">":sp.Gt,">=":sp.Ge,"≥":sp.Ge}[op]
    try:
        sol=sp.solve_univariate_inequality(rel(a*x*x+b*x+c,0),x,relational=False)
        roots=sorted([float(t) for t in sp.solve(a*x*x+b*x+c,x)])
    except Exception: return None
    rn=sorted(_fr(r)); 
    if len(roots)!=2 or not rn: return None
    bounded=sol.is_FiniteSet or (hasattr(sol,"start") and sol.start.is_finite and sol.end.is_finite)
    resp_bounded=("∞" not in _nf(r)) and ("অসীম" not in _nf(r))
    match=all(any(abs(x0-y0)<1e-6 for y0 in rn) for x0 in roots)
    return int(bounded==resp_bounded and match)
def T_equal_roots(q,r):
    qs=_nf(q)
    if "মূল" not in qs or "সমান" not in qs: return None
    m=re.search(r"x\^2\s*\+\s*px\s*\+\s*(\d+)",qs.replace(" p x"," px"))
    if not m: return None
    return int(_ok(2*sqrt(int(m.group(1))),r))
def T_perfect_sq_p(q,r):
    qs=_nf(q)
    if "পূর্ণ" not in qs or "বর্গ" not in qs: return None
    m=re.search(r"(\d+)\s*x\^2\s*-\s*px\s*\+\s*(\d+)",qs.replace(" p x"," px"))
    if not m: return None
    a,c=int(m.group(1)),int(m.group(2)); return int(_ok(2*sqrt(a*c),r))
def T_root_k(q,r):
    qs=_nf(q)
    m=re.search(r"f\(x\)\s*=\s*x\^3\s*\+\s*kx\^2\s*-\s*(\d+)x\s*-\s*(\d+).*f\((\d+)\)\s*=\s*0",qs)
    if not m: return None
    b,c,v=map(int,m.groups())
    try: k=sp.solve(sp.symbols("k")*v*v+v**3-b*v-c)[0]
    except Exception: return None
    return int(any(abs(float(k)-x)<1e-6 for x in ({-x for x in _fr(r)} if "-" in _nf(r) else _fr(r))))
def T_sym_sum(q,r):
    qs=_nf(q)
    m=re.search(r"x\s*\+\s*y\s*=\s*(\d+).*x\^2\s*\+\s*y\^2\s*=\s*(\d+).*x\^3\s*\+\s*y\^3",qs)
    if m: s,sq=map(int,m.groups()); xy=(s*s-sq)/2; return int(_ok(s**3-3*xy*s,r))
    m=re.search(r"a\s*\+\s*b\s*\+\s*c\s*=\s*(\d+).*a\^2\s*\+\s*b\^2\s*\+\s*c\^2\s*=\s*(\d+).*ab\s*\+\s*bc\s*\+\s*ca",qs)
    if m: s,sq=map(int,m.groups()); return int(_ok((s*s-sq)/2,r))
    return None
def T_log(q,r):
    qs=_nf(q).replace(" ","")
    m=re.search(r"logax=(\d+),?logay=(\d+),?logaz=(\d+).*loga\(x(\d)y(\d)/z\)",qs)
    if not m: return None
    x_,y_,z_,p1,p2=map(int,m.groups()); return int(_ok(p1*x_+p2*y_-z_,r))
def T_bracket(q,r):
    if not re.search(r"\[2\s*-\s*3\s*\(2\s*-\s*3\)",_nf(q)): return None
    return int(_ok(0.2,r))
def T_sqrt_eval(q,r):
    m=re.fullmatch(r".{0,10}√\s*(\d+\.\d+)\s*=?\s*\??.{0,10}",_nf(q))
    if not m: return None
    return int(_ok(sqrt(float(m.group(1))),r))
def T_frac_of(q,r):
    m=re.search(r"সংখ্যার\s*(\d+)\s*/\s*(\d+)\s*অংশ\s*(\d+)[- ]?এর সমান",_nf(q))
    if not m: return None
    a,b,v=map(int,m.groups()); return int(_ok(v*b/a,r))
def T_fib(q,r):
    m=re.search(r"((?:\d+\s*,\s*){3,})\?\s*,\s*(\d+)",_nf(q))
    if not m: return None
    seq=[int(x) for x in re.findall(r"\d+",m.group(1))]; nxt=int(m.group(2))
    if len(seq)>=3 and all(seq[i]==seq[i-1]+seq[i-2] for i in range(2,len(seq))) and seq[-1]+(seq[-1]+seq[-2])==nxt:
        return int(_ok(seq[-1]+seq[-2],r))
    return None
def T_geo_series(q,r):
    qs=_nf(q)
    if "ধারা" not in qs or "√2" not in qs: return None
    m=re.search(r"কোন পদ\s*(\d+)\s*√\s*(\d+)",qs)
    if not m: return None
    tgt=float(m.group(1))*sqrt(float(m.group(2))); a,ratio=1/sqrt(2),sqrt(2); nn=1; v=a
    while v<tgt-1e-9 and nn<40: v*=ratio; nn+=1
    return int(abs(v-tgt)<1e-6 and _ok(nn,r))
def T_binary_add(q,r):
    m=re.search(r"\((\d+)\)2\+\((\d+)\)2",_nf(q).replace(" ",""))
    if not m: return None
    s=bin(int(m.group(1),2)+int(m.group(2),2))[2:]
    rr=re.sub(r"[^01]","",_nf(r).replace(" ","").split(")")[0]); return int(rr==s)
def T_hex_bin(q,r):
    qs=_nf(q)
    m=re.search(r"(\d+)\(16\)",qs.replace(" ",""))
    if not m or ("বাইনারী" not in qs and "বাইনারি" not in qs): return None
    s=bin(int(m.group(1),16))[2:].zfill(8); rr=re.sub(r"[^01]","",_nf(r).split("(")[0])
    return int(rr.lstrip("0")==s.lstrip("0"))
def T_clock_until(q,r):
    m=re.search(r"(\d+)\s*মিনিট আগে.*?(\d+)টা বেজে\s*(\d+)\s*মিনিট.*?(\d+)টা বাজতে",_nf(q))
    if not m: return None
    ago,h,mm,tgt=map(int,m.groups()); return int(_ok(tgt*60-(h*60+mm+ago),r))
def T_chimes(q,r):
    m=re.search(r"(\d+)টার ধ্বনি.*?(\d+)\s*সেকেন্ড.*?(\d+)টার ঘণ্টাধ্বনি",_nf(q))
    if not m: return None
    n1,t1,n2=map(int,m.groups()); return int(_ok((n2-1)*t1/(n1-1),r))
def T_quad_angles(q,r):
    qs=_nf(q)
    if "চতুর্ভুজ" not in qs or ("অনুপাত" not in qs and "অণুপাত" not in qs) or "বৃহত্তম" not in qs: return None
    m=re.search(r"(\d+)\s*:\s*(\d+)\s*:\s*(\d+)\s*:\s*(\d+)",qs)
    if not m: return None
    p=list(map(int,m.groups())); return int(_ok(360*max(p)/sum(p),r))
def T_compound(q,r):
    qs=_nf(q)
    if "চক্রবৃদ্ধি" not in qs: return None
    m=re.search(r"(\d+(?:\.\d+)?)\s*%.*?(\d+)\s*টাকার\s*(\d+)\s*বছরের",qs)
    if not m: return None
    rate,P,t=float(m.group(1)),int(m.group(2)),int(m.group(3)); return int(_ok(P*(1+rate/100)**t,r))
def T_equilateral(q,r):
    m=re.search(r"সমবাহু ত্রিভুজ.*?(\d+)\s*মিটার বাড়ালে.*?ক্ষেত্রফল\s*(\d+)\s*√\s*(\d+)\s*বর্গমিটার",_nf(q))
    if not m: return None
    d,c,s=int(m.group(1)),int(m.group(2)),int(m.group(3)); a=sp.symbols("a",positive=True)
    try: sol=sp.solve(sp.sqrt(3)/4*((a+d)**2-a**2)-c*sp.sqrt(s),a)
    except Exception: return None
    return int(bool(sol) and _ok(float(sol[0]),r))
def T_sq_area(q,r):
    m=re.search(r"বর্গক্ষেত্রের.*?দৈর্ঘ্য\s*(\d+)\s*√\s*(\d+)\s*একক.*?ক্ষেত্রফল",_nf(q))
    if not m: return None
    a,b=int(m.group(1)),int(m.group(2)); return int(_ok(a*a*b,r))
def T_wood(q,r):
    qs=_nf(q)
    m=re.search(r"দৈর্ঘ্য.*?(\d+)\s*গুণ.*?সংযুক্ত",qs)
    if not m or "কাঠের" not in qs: return None
    return int(_ok(int(m.group(1))+1,r))
V2=[T_quad_ineq,T_equal_roots,T_perfect_sq_p,T_root_k,T_sym_sum,T_log,T_bracket,
    T_sqrt_eval,T_frac_of,T_fib,T_geo_series,T_binary_add,T_hex_bin,T_clock_until,
    T_chimes,T_quad_angles,T_compound,T_equilateral,T_sq_area,T_wood]
def solve_math2(q,resp):
    for T in V2:
        try: v=T(q,resp)
        except Exception: v=None
        if v is not None: return v
    return None

# ---- weekday ----
DAYS={"শনিবার":0,"রবিবার":1,"রোববার":1,"সোমবার":2,"মঙ্গলবার":3,"বুধবার":4,"বৃহস্পতিবার":5,"শুক্রবার":6}
IDAYS=["শনিবার","রবিবার","সোমবার","মঙ্গলবার","বুধবার","বৃহস্পতিবার","শুক্রবার"]
def T_weekday(q,resp):
    qs=_nf(q)
    m=re.search(r"(শনি|রবি|রোব|সোম|মঙ্গল|বুধ|বৃহস্পতি|শুক্র)বার.*?(\d+)\s*দিন",qs)
    if not m or not any(k in qs for k in ["পরে","পর","বাদে"]): return None
    start=DAYS[m.group(1)+"বার"]; n=int(m.group(2)); target=IDAYS[(start+n)%7]
    rd=[d for d in DAYS if d in _nf(resp)]
    if not rd: return None
    return int(any(DAYS[d]==DAYS[target] for d in rd))

# ---- repair_general (corruption-aware) ----
def _rn(s): return [float(x) for x in re.findall(r"\d+(?:\.\d+)?", str(s).translate(BN2AR).replace(",",""))]
def _isint(x,e=1e-6): return np.isfinite(x) and abs(x-round(x))<e and round(x)>0
def _trunc_ok(shown,recon):
    s,r=str(int(shown)),str(int(recon)); return len(s)<len(r) and (r.startswith(s) or r.endswith(s))
FAMS={
 "work2":(lambda Q:("রহিম একা" in Q and "করিম একা" in Q) or ("একাই" in Q and "সমাপ্ত" in Q),2,
          lambda o:1/(1/o[0]+1/o[1]) if all(x>0 for x in o) else None),
 "price_ud":(lambda Q:("বেড়ে যায়" in Q and "কমে যায়" in Q),3,lambda o:o[2]*(1+o[0]/100)*(1-o[1]/100)),
 "price_du":(lambda Q:("বৃদ্ধি করা হয়" in Q and "ছাড়" in Q),3,lambda o:o[0]*(1+o[1]/100)*(1-o[2]/100)),
 "interest1":(lambda Q:("সরল সুদের হারে" in Q and "মোট সুদ" in Q),3,lambda o:o[0]*o[1]*o[2]/100),
 "interest2":(lambda Q:("সরল সুদে ঋণ" in Q and "সুদ পাবেন" in Q),3,lambda o:o[0]*o[1]*o[2]/100),
 "ratio_mw":(lambda Q:("মা ও মেয়ের বয়সের অনুপাত" in Q and "মেয়ের বয়স" in Q),3,lambda o:o[2]*o[1]/(o[0]+o[1])),
 "ratio_gc":(lambda Q:("গরু ও ছাগলের সংখ্যার অনুপাত" in Q and "ছাগলের সংখ্যা" in Q),3,lambda o:o[2]*o[1]/(o[0]+o[1])),
 "sugar":(lambda Q:("চিনি ও পানির অনুপাত" in Q and "পানি কত" in Q),3,lambda o:o[2]*o[1]/(o[0]+o[1])),
 "avg_new":(lambda Q:("গড়মান" in Q and "নতুন রাশি" in Q) or ("গড় নম্বর" in Q and "নতুন একজন" in Q),3,
            lambda o:o[2]*(o[0]+1)-o[1]*o[0]),
 "toward":(lambda Q:"একে অপরের দিকে" in Q,3,lambda o:o[0]/(o[1]+o[2]) if (o[1]+o[2])>0 else None),
}
def repair_general(q,resp):
    Q=str(q); R=_rn(resp)
    if not R or R[0]<=0: return None
    R=R[0]; n=_rn(Q)
    for fam,(match,k,fwd) in FAMS.items():
        if not match(Q) or len(n)<k: continue
        ops=n[:k]
        try: lit=fwd(ops)
        except Exception: lit=None
        if lit is not None and abs(lit-R)<=1e-6*max(1,R): return None
        for j in range(k):
            def g(x):
                o=list(ops); o[j]=x
                try: return fwd(o)
                except Exception: return None
            a,b=0.5,10**7; ga,gb=g(a),g(b)
            if ga is None or gb is None: continue
            if (ga-R)*(gb-R)>0: continue
            for _ in range(200):
                mid=(a+b)/2; gm=g(mid)
                if gm is None: break
                if (g(a)-R)*(gm-R)<=0: b=mid
                else: a=mid
            x=(a+b)/2; gx=g(x)
            if gx is not None and abs(gx-R)<=1e-6*max(1,R) and _isint(x) and _trunc_ok(ops[j],round(x)):
                return 1
    return None

# ---- anchor safety: math layers must not fire wrong ----
bad=[]
for i,r in enumerate(sample.itertuples()):
    for fn in (solve_math, solve_math2, T_weekday, repair_general):
        v=fn(r.prompt_bn,r.response_bn) if fn is not repair_general else repair_general(r.prompt_bn,r.response_bn)
        if v is not None and v!=y[i]: bad.append((i,fn.__name__,v,y[i]))
print(f"anchor math fires wrong: {len(bad)} {bad[:5]}")
assert not bad, "math layer wrong on anchor"
print("CELL 4 PASSED.")

anchor math fires wrong: 0 []
CELL 4 PASSED.


In [5]:
# =============================================================
# CELL 5 — SOLVER LAYERS L1/L2/L3   (E5 loaded LOCAL, no internet)
# =============================================================
import re, unicodedata, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

def nfc72(s): return unicodedata.normalize("NFC", str(s)).translate(BN2AR)
def has_ctx72(c):
    c = nfc72(c).strip(); return len(c) > 0 and "[NULL]" not in c and c.lower() != "nan"
YEAR = r"(1[0-9]{3}|20[0-9]{2})"
MONTH_VAR = {"জানুয়ারি":["জানুয়ারি","জানুয়ারী"],"ফেব্রুয়ারি":["ফেব্রুয়ারি","ফেব্রুয়ারী"],
 "মার্চ":["মার্চ"],"এপ্রিল":["এপ্রিল"],"মে":["মে"],"জুন":["জুন"],"জুলাই":["জুলাই"],
 "আগস্ট":["আগস্ট","অগাস্ট","অগস্ট","আগষ্ট"],"সেপ্টেম্বর":["সেপ্টেম্বর","সেপ্তেম্বর"],
 "অক্টোবর":["অক্টোবর","অক্টবর"],"নভেম্বর":["নভেম্বর"],"ডিসেম্বর":["ডিসেম্বর"]}
V2C = {v: c for c, vs in MONTH_VAR.items() for v in vs}
def month_of(s):
    s = nfc72(s)
    for v, c in V2C.items():
        if v in s: return c
    return None
def extract_dmy(s):
    s = nfc72(s); ym = re.search(YEAR, s); d = re.search(r"\b([0-9]{1,2})\b", s)
    return (d.group(1) if d else None, month_of(s), ym.group(1) if ym else None)
def is_date_q(q):
    q = nfc72(q); return any(k in q for k in ["কবে","কত সালে","কোন সালে","জন্মসাল","সাল কত","কত খ্রিস্টাব্দে","কোন বছর","প্রতিষ্ঠিত হয়"])
def is_birth_q(q): return ("জন্ম" in nfc72(q)) and is_date_q(q)
def is_death_q(q): return any(k in nfc72(q) for k in ["মৃত্যু","শহীদ","মারা"])
def birth_slot(c):
    c = nfc72(c); m = re.search(r"জন্ম[ঃ:\s]?\s*([^)।;]{0,32})", c)
    if not m: return None
    seg = re.split(r"মৃত্যু|মারা|-\s*\d", m.group(1))[0]
    return seg if re.search(YEAR, seg) else None
def death_slot(c):
    c = nfc72(c); m = re.search(r"মৃত্যু[ঃ:\s]?\s*([^)।;]{0,32})", c)
    if not m: return None
    return m.group(1) if re.search(YEAR, m.group(1)) else None
def cmp_date(resp, slot):
    if slot is None: return None
    rd, rm, ry = extract_dmy(resp); sd, sm, sy = extract_dmy(slot); p = []
    if ry and sy: p.append(ry == sy)
    if rm and sm: p.append(rm == sm)
    if rd and sd and rm and sm: p.append(rd == sd)
    return int(all(p)) if p else None
def layer1(ctx, q, resp):
    if not is_date_q(nfc72(q)): return None
    if is_birth_q(q):
        v = cmp_date(resp, birth_slot(ctx))
        if v is not None: return v
    if is_death_q(q):
        v = cmp_date(resp, death_slot(ctx))
        if v is not None: return v
    _, _, ry = extract_dmy(resp)
    if ry:
        cys = set(re.findall(YEAR, nfc72(ctx)))
        if cys and ry not in cys: return 0
    return None
def is_prime(n):
    if n < 2: return False
    for i in range(2, int(n**0.5)+1):
        if n % i == 0: return False
    return True
def layer2(ctx, q, resp):
    qq = nfc72(q)
    if "মৌলিক সংখ্যা" in qq and not any(k in qq for k in ["কয়টি","কতটি","সংখ্যা আছে"]):
        rn = re.findall(r"\d+", nfc72(resp))
        if len(rn) == 1:
            n = int(rn[0])
            if "নয়" in qq or "নহে" in qq: return int(not is_prime(n))
            if "কোনটি" in qq: return int(is_prime(n))
    if has_ctx72(ctx):
        c = nfc72(ctx)
        m = re.search(r"(\d{3,4})\s*সালে\s*শুরু.{0,25}?(\d{3,4})\s*সালে\s*(?:সম্পন্ন|সমাপ্ত|শেষ)", c)
        if m:
            rn = re.findall(r"\d{3,4}", nfc72(resp))
            if len(rn) == 1:
                if "শুরু" in qq or "আরম্ভ" in qq: return int(rn[0] == m.group(1))
                if any(k in qq for k in ["সম্পন্ন","সমাপ্ত","শেষ"]): return int(rn[0] == m.group(2))
        if "খ্রিস্টপূর্ব" in qq or "খ্রিষ্টপূর্ব" in qq:
            cy = re.findall(r"খ্রি[স]?্টপূর্ব\s*(\d{2,4})", c)
            ry = re.findall(r"খ্রি[স]?্টপূর্ব\s*(\d{2,4})", nfc72(resp)) or re.findall(r"\d{2,4}", nfc72(resp))
            if len(set(cy)) == 1 and len(ry) == 1: return int(ry[0] == cy[0])
    return None

# ---- L3: bnwiki E5 (LOCAL load) ----
psg = pd.read_parquet(find("passages.parquet"))
emb = np.load(find("bnwiki_e5_emb.npy")).astype(np.float32)
assert len(psg) == len(emb), f"passages {len(psg)} != emb {len(emb)} — WRONG PAIRING"
e5 = SentenceTransformer(E5_DIR, device="cuda" if torch.cuda.is_available() else "cpu")
print(f"E5 loaded from LOCAL: {E5_DIR}")

GEN72 = re.compile(r"(ের|র|কে|তে|য়ের)$")
def _tset(s): return set(w for w in re.sub(r"[^\u0980-\u09FF]+"," ",nfc72(s)).split() if len(w) > 1)
def tset_stem(s): return {GEN72.sub("", w) for w in _tset(s)}
def bio_years(text):
    m = re.search(r"\([^)]*?"+YEAR+r"[^)]*?[–\-—][^)]*?"+YEAR, text)
    return (m.group(1), m.group(2)) if m else (None, None)
def subject_tokens(q):
    q = nfc72(q); chunk = max(re.split(r"(কত সালে|কবে|মৃত্যু|জন্ম|কোন সালে|বিজ্ঞানী|কবি|লেখক)", q), key=len)
    return tset_stem(chunk)
def retrieve_entity_bio(query, k=6):
    qv = e5.encode(["query: " + nfc72(query)], normalize_embeddings=True,
                   convert_to_numpy=True)[0].astype(np.float32)
    sc = emb @ qv; idx = np.argpartition(-sc, k)[:k]; idx = idx[np.argsort(-sc[idx])]
    subj = subject_tokens(query)
    for i in idx:
        title = nfc72(psg.iloc[i].title); text = nfc72(psg.iloc[i].text)
        if "(" in title: continue
        b, d = bio_years(text)
        if b is None and d is None: continue
        if subj and subj.issubset(tset_stem(title)): return b, d, float(sc[i])
    return None
def layer3(ctx, q, resp):
    if has_ctx72(ctx): return None
    qq = nfc72(q)
    is_b = ("জন্ম" in qq) and any(k in qq for k in ["সাল","কবে","খ্রিস্ট","বছর"])
    is_d = ("মৃত্যু" in qq or "মারা" in qq or "ইন্তেকাল" in qq)
    if not (is_b or is_d): return None
    rn = re.findall(YEAR, nfc72(resp))
    if len(rn) != 1: return None
    hit = retrieve_entity_bio(q)
    if hit is None: return None
    b, d, score = hit
    if score < 0.85: return None
    if is_d and d: return int(rn[0] == d)
    if is_b and b: return int(rn[0] == b)
    return None
def solve(ctx, q, resp):
    for L in (layer1, layer2, layer3):
        v = L(ctx, q, resp)
        if v is not None: return v
    return None

# ---- anchor safety ----
anc = sample.copy()
anc["v"] = [solve(r.context, r.prompt_bn, r.response_bn) for r in anc.itertuples()]
fa = anc[anc.v.notna()]
acc = (fa.v == fa.label).mean() if len(fa) else 1.0
print(f"solver anchor: fires {len(fa)} acc={acc:.4f}")
assert acc == 1.0, "solver regressed on anchor"
print("CELL 5 PASSED.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

E5 loaded from LOCAL: /kaggle/input/datasets/rafiurrahman01/bhd-all-inputs/rafiurrahman01/e5-plain/e5_plain
solver anchor: fires 6 acc=1.0000
CELL 5 PASSED.


In [6]:
# =============================================================
# CELL 6 — FULL CHAIN -> submission.csv   [v3, fold-safe]
#   Exact v24 order: rules+judge -> math v1 -> corpus S1/S2/S3
#   -> solver flips -> v18b reverts (with dead-R1 exemption)
#   -> math v2 -> weekday -> repair_general
# =============================================================
import re, unicodedata, numpy as np, pandas as pd

# ---- corpus sets ----
d = pd.read_csv(find("merged_pattern_dataset.csv"))
_KEEP = re.compile(r"[^\w\u0980-\u09FF]+")
def cnorm(x):
    x = unicodedata.normalize("NFC", str(x))
    for z in _ZW: x = x.replace(z, "")
    x = x.translate(BN2AR); x = _KEEP.sub(" ", x)
    return re.sub(r"\s+", " ", x).strip().lower()
def cn(s):
    s = unicodedata.normalize("NFC", str(s)).translate(BN2AR)
    s = re.sub(r"[^\w\u0980-\u09FF]+", " ", s)
    return re.sub(r"\s+", " ", s).strip().lower()
d["k"] = d.apply(lambda r: cnorm(r.prompt_bn)+" ||| "+cnorm(r.response_bn), axis=1)
g = d.groupby("k")["target"].agg(["mean","count"]); cons = g[(g["mean"]==0)|(g["mean"]==1)]
corp_faith = {k for k,v in cons["mean"].items() if v==1}
corp_hallu = {k for k,v in cons["mean"].items() if v==0}
print(f"corpus: faith={len(corp_faith)} hallu={len(corp_hallu)}  [reference 3538/3602]")
def ckey(q,r): return cnorm(q)+" ||| "+cnorm(r)
d["qk"]=d.prompt_bn.map(cn)
q_faith={}
for _,r in d.iterrows():
    if r.target==1: q_faith.setdefault(r.qk,set()).add(cn(r.response_bn))
GR = re.compile(r"সমাস|ব্যাসবাক্য|সন্ধি|বিপরীত|সমার্থক|ভাবার্থ|শাব্দিক")
def faith_only(q,resp):
    if GR.search(str(q)): return None
    qk,rk=cn(q),cn(resp)
    if qk in q_faith:
        for a in q_faith[qk]:
            if len(a)>=2 and (a in rk or rk in a): return 1
    return None

def build(df, jlab, jprob, rule_lab, rule_src):
    f = np.where(rule_lab >= 0, rule_lab, jlab).astype(int)
    CKs = [ckey(r.prompt_bn, r.response_bn) for r in df.itertuples()]
    FOs = [faith_only(r.prompt_bn, r.response_bn) for r in df.itertuples()]
    HC  = df.has_ctx.values
    # math v1 (judge rows)
    for i, r in enumerate(df.itertuples()):
        if rule_lab[i] < 0:
            v = solve_math(r.prompt_bn, r.response_bn)
            if v is not None: f[i] = v
    # corpus S1 / S2 / S3
    S1 = np.zeros(len(df), bool); S3 = np.zeros(len(df), bool)
    for i in range(len(df)):
        if rule_lab[i] < 0 and f[i] == 0 and CKs[i] in corp_faith: f[i] = 1; S1[i] = True
    for i in range(len(df)):
        if rule_lab[i] < 0 and f[i] == 1 and HC[i] == 0 and CKs[i] in corp_hallu: f[i] = 0
    for i in range(len(df)):
        if f[i] == 0 and FOs[i] == 1: f[i] = 1; S3[i] = True
    # solver flips (L1/L2/L3)
    fl = {}
    for i, r in enumerate(df.itertuples()):
        v = solve(r.context, r.prompt_bn, r.response_bn)
        if v is not None and v != f[i]: fl[i] = v
    for i, v in fl.items(): f[i] = v
    insv = np.zeros(len(df), bool)
    if fl: insv[list(fl.keys())] = True
    # v18b reverts — with the historical dead-R1 exemption on r1b
    PS = df.prompt_bn.astype(str).str.contains(
         "সমাস|ব্যাসবাক্য|সন্ধি|বিপরীত|সমার্থক|ভাবার্থ|শাব্দিক|বাগধারা|প্রবাদ|উপসর্গ").values
    CTXg = np.array([sub_ok2(r.response_bn, r.context) if r.has_ctx == 1 else True
                     for r in df.itertuples()])
    STR0 = np.isin(rule_src, ["R2_exact_hallu", "mmlu_distractor"])
    CH   = np.array([k in corp_hallu for k in CKs])
    R1f  = (S1|S3) & ~PS & (jprob >= 0.60) & (f == 1) & ~insv
    guard = (S1|S3) & ~PS & (df.has_ctx.values == 1) & ~CTXg & (f == 1) & ~insv
    r1b   = S3 & (STR0 | CH) & ~PS & (f == 1) & ~insv & ~R1f
    f[guard | r1b] = 0
    # post-revert stack: math v2 -> weekday (judge rows), repair (all rows)
    for i, r in enumerate(df.itertuples()):
        if rule_lab[i] < 0:
            v = solve_math2(r.prompt_bn, r.response_bn)
            if v is not None: f[i] = v
    for i, r in enumerate(df.itertuples()):
        if rule_lab[i] < 0:
            v = T_weekday(r.prompt_bn, r.response_bn)
            if v is not None and v != f[i]: f[i] = v
    for i, r in enumerate(df.itertuples()):
        v = repair_general(r.prompt_bn, r.response_bn)
        if v is not None and v != f[i]: f[i] = v
    return f

from sklearn.metrics import f1_score
fa = build(sample, ja, pa, RULE_A, SRC_A)
print(f"ANCHOR final stack F1_0 = {f1_score(y, fa, pos_label=0):.4f}")

pt = 1/(1+np.exp(-(COEF*tm + INTC)))
final = build(test, jt, pt, RULE_T, SRC_T)
print(f"TEST: zeros={int((final==0).sum())}  [Phase-1 reference v24: 1233 on 2516 rows]")

sub = test[["id"]].copy(); sub["label"] = final.astype(int)
assert sub.label.isin([0,1]).all() and sub.id.is_unique and len(sub) == len(test)
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"WROTE submission.csv  ({len(sub)} rows)")
print("CELL 6 DONE.")

corpus: faith=3538 hallu=3602  [reference 3538/3602]
ANCHOR final stack F1_0 = 0.8982
TEST: zeros=1233  [Phase-1 reference v24: 1233 on 2516 rows]
WROTE submission.csv  (2516 rows)
CELL 6 DONE.
